# SETUP

In [1]:
import os
import jax
import jax.numpy as jnp
import numpy as np
import gc
import time
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [2]:
print("JAX version:", jax.__version__)
print("JAX XLA backend platform:", jax.lib.xla_bridge.get_backend().platform)
print("Available devices:", jax.devices())
print("Default device:", jax.default_backend())

JAX version: 0.4.30
JAX XLA backend platform: gpu
Available devices: [cuda(id=0)]
Default device: gpu


In [3]:
x = jnp.ones((1000, 1000))
y = jnp.dot(x, x)
print("Computation device:", x.devices())
print("Computation device:", y.devices())

x_np = np.ones((1000, 1000))
y_np = np.dot(x, x)
x_jnp = jnp.array(x_np)
y_jnp = jnp.dot(x_np, x_np)
print("Computation device:", x_jnp.devices())
print("Computation device:", y_jnp.devices())

Computation device: {cuda(id=0)}
Computation device: {cuda(id=0)}
Computation device: {cuda(id=0)}
Computation device: {cuda(id=0)}


In [3]:
z = jnp.ones((5000, 5000))
print("Computation device:", z.devices())

Computation device: {cuda(id=0)}


In [4]:
# z_cpu = jax.device_put(z, device=jax.devices("cpu")[0])
# print("Computation device:", z_cpu.devices())

del z
gc.collect()

10

In [5]:
a = jnp.ones((5000, 5000))
print("Computation device:", a.devices())

Computation device: {cuda(id=0)}


In [7]:
b = jnp.ones((5000, 5000))
print("Computation device:", b.devices())

Computation device: {cuda(id=0)}


In [15]:
if True: raise Exception

Exception: 

# JIT

In [ ]:
# input tensors
A_np = np.random.rand(100, 100)
B_np = np.random.rand(100, 100)

A_jax = jnp.array(A_np)
B_jax = jnp.array(B_np)

In [ ]:
# function for native JIT
def jax_function(A, B):
	C = B
	for _ in range(50):
		C = A @ C
		C = (C + A + B) / 3
		C = C * A * B
		C = jnp.transpose(C)
		C = jnp.linalg.pinv(C)
	d = jnp.linalg.det(C)
	return d

# trace & warm up
jax_function_jit = jax.jit(jax_function)
for _ in range(100): yt = jax_function_jit(A_jax, B_jax).block_until_ready()

In [ ]:
# no JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"raw avg time: {sum(runtimes)/len(runtimes):.6f} seconds")

In [ ]:
# JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_jit(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"JIT avg time: {sum(runtimes)/len(runtimes):.6f} seconds")

# AUTOGRAD

In [ ]:
# grad, trace & warm up
jax_function_grad = jax.grad(jax_function)

jax_function_grad_jit = jax.jit(jax_function_grad)
for _ in range(100): yt = jax_function_grad_jit(A_jax, B_jax).block_until_ready()

In [ ]:
# no JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_grad(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"raw avg time: {sum(runtimes)/len(runtimes):.6f} seconds")

In [ ]:
# JIT

runtimes = list()
for _ in range(100):
	start = time.time()
	y = jax_function_grad_jit(A_jax, B_jax).block_until_ready()
	end = time.time()
	runtimes.append(end - start)
print(f"JIT avg time: {sum(runtimes)/len(runtimes):.6f} seconds")